<div text-align="center">
  <img src="https://raw.githubusercontent.com/FarnoushRJ/MambaLRP/main/assets/MambaLRP_logo.jpeg" width="1000"/>
</div>


<div text-align="center"><h1>🐍 MambaLRP is here! 🎉</h1>

Clone the repository and install MambaLRP.

In [1]:
# Cell 1: Clone
!git clone https://github.com/AdamBosch/MambaLRP.git || echo "Already cloned"

fatal: destination path 'MambaLRP' already exists and is not an empty directory.
Already cloned


In [2]:
# Cell 5: Install MambaLRP
!pip install ./MambaLRP --no-deps -q

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [17 lines of output]
      Traceback (most recent call last):
        File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
        File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/tmp/pip-build-env-bbyw2cj0/overlay/local/lib/python3.10/dist-packages/setuptools/build_meta.py", line 333, in get_requires_for_build_wheel
          return self._get_build_requires(config_settings, requirements=[])
        File "/tmp/pip-build-env-bbyw2cj0/o

In [3]:
import datasets
import pyarrow
import fsspec
import huggingface_hub

print(datasets.__version__)
print(pyarrow.__version__)
print(fsspec.__version__)
print(huggingface_hub.__version__)

2.18.0
14.0.2
2026.4.0
0.36.2


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import transformers
print(transformers.__version__)  # should be 4.x

import torch
print(torch.__version__)         # 2.1.1+cu118

import causal_conv1d
import mamba_ssm
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
print("All imports OK")

4.57.6
2.2.2+cu118
All imports OK


In [5]:
import sys

from mamba_lrp.model.mamba_huggingface import ModifiedMambaForCausalLM
from mamba_lrp.model.utils import *
from mamba_lrp.lrp.utils import relevance_propagation, relevance_propagation_ig_lrp
from mamba_lrp.dataset.general_dataset import get_emotion_dataset
import torch
import numpy as np

## Load model

Load model and tokenizer.

In [6]:
import torch
import gc

# 1. Delete the variables holding the references
# Note: Ensure you are actually deleting the correct names
if 'R' in locals(): del R
if 'attr' in locals(): del attr
if 'embeddings' in locals(): del embeddings
if 'morf_probs' in locals(): del morf_probs

# 2. Force Python to clear unused references from RAM
gc.collect()

# 3. Force PyTorch to release its internal cache back to the OS
torch.cuda.empty_cache()

# 4. Optional: Check how much is actually used vs reserved
print(f"Actually Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved (Cached): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Actually Used: 0.00 GB
Reserved (Cached): 0.00 GB


In [7]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

model = MambaForCausalLM.from_pretrained(
    "state-spaces/mamba-130m-hf",
    use_cache=True
)

model.lm_head = torch.nn.Linear(768, 6, bias=True)

model.config.pad_token_id = tokenizer.pad_token_id

In [11]:
import torch
from torch.utils.data import DataLoader
from transformers import AdamW
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm


def train_emotion_local(model, tokenizer, device):

    # =========================================================
    # 1. LOAD DATASETS PROPERLY
    # =========================================================
    train_dataset = get_emotion_dataset(tokenizer, split='train')
    val_dataset = get_emotion_dataset(tokenizer, split='validation')
    test_dataset = get_emotion_dataset(tokenizer, split='test')

    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")

    # =========================================================
    # 2. DATALOADERS
    # =========================================================
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    # =========================================================
    # 3. FREEZE MODEL
    # =========================================================
    # for param in model.parameters():
    #     param.requires_grad = False

    # Replace classification head
    model.lm_head = torch.nn.Linear(768, 6, bias=True).to(device)

    optimizer = AdamW(model.parameters(), lr=7e-5)

    criterion = torch.nn.CrossEntropyLoss()

    scheduler = LinearLR(
        optimizer,
        start_factor=0.5,
        total_iters=5
    )

    # =========================================================
    # 4. TRAINING
    # =========================================================
    best_val_loss = float('inf')
    max_epochs = 10
    
    patience = 3
    no_improve_count = 0

    for epoch in range(max_epochs):

        # ---------------- TRAIN ----------------
        model.train()

        total_train_loss = 0

        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1} [Train]"
        )

        for batch in pbar:

            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids)

            pad_id = tokenizer.pad_token_id
            seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
            logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
            
            loss = criterion(logits, labels)

            loss.backward()

            optimizer.step()

            total_train_loss += loss.item()

            pbar.set_postfix({
                'loss': f"{loss.item():.4f}"
            })

        avg_train_loss = total_train_loss / len(train_loader)

        # ---------------- VALIDATION ----------------
        model.eval()

        total_val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for batch in val_loader:

                input_ids = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids)

                pad_id = tokenizer.pad_token_id
                seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
                logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
                
                loss = criterion(logits, labels)

                total_val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_accuracy = correct / total

        print(
            f"Epoch {epoch + 1} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {val_accuracy * 100:.2f}%"
        )

        scheduler.step()

        # ---------------- EARLY STOPPING ----------------
        if avg_val_loss < best_val_loss:

            best_val_loss = avg_val_loss

            torch.save(
                model.state_dict(),
                'mamba_emotion_weights.pt'
            )

            print("Saved best model.")

        else:
            no_improve_count += 1
            print(f"No improvement ({no_improve_count}/{patience})")
            if no_improve_count >= patience:
                print("Early stopping.")
                break

    # =========================================================
    # 5. FINAL TEST EVALUATION
    # =========================================================
    print("\nEvaluating on Test Set...")

    model.load_state_dict(
        torch.load('mamba_emotion_weights.pt')
    )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids)

            pad_id = tokenizer.pad_token_id
            seq_lengths = (input_ids != pad_id).sum(dim=1) - 1
            logits = outputs.logits[torch.arange(input_ids.size(0)), seq_lengths]
            
            preds = torch.argmax(logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_accuracy = correct / total

    print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")


# =========================================================
# EXECUTION
# =========================================================
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model.to(device)

train_emotion_local(model, tokenizer, device)

Train size: 16000
Validation size: 2000
Test size: 2000


/home/jovyan/.local/lib/python3.10/site-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 1 [Train]: 100%|██████████| 500/500 [02:17<00:00,  3.64it/s, loss=0.2602]


Epoch 1 | Train Loss: 0.6336 | Val Loss: 0.1687 | Val Acc: 93.35%
Saved best model.


Epoch 2 [Train]: 100%|██████████| 500/500 [02:17<00:00,  3.63it/s, loss=0.0737]


Epoch 2 | Train Loss: 0.1221 | Val Loss: 0.1280 | Val Acc: 94.10%
Saved best model.


...

Epoch 3 | Train Loss: 0.0797 | Val Loss: 0.1415 | Val Acc: 93.75%
No improvement (1/3)


Epoch 4 [Train]: 100%|██████████| 500/500 [02:18<00:00,  3.62it/s, loss=0.0427]


Epoch 4 | Train Loss: 0.0565 | Val Loss: 0.1405 | Val Acc: 94.10%
No improvement (2/3)


Epoch 5 [Train]: 100%|██████████| 500/500 [02:17<00:00,  3.62it/s, loss=0.0541]


Epoch 5 | Train Loss: 0.0398 | Val Loss: 0.1943 | Val Acc: 93.60%
No improvement (3/3)
Early stopping.

Evaluating on Test Set...
Final Test Accuracy: 92.80%


In [7]:
!pip install gdown

# Import gdown
import gdown

# Define the file ID and the destination file name
file_id = '17J33ZY5lDiPneHy0ay29J5uxL-F0KAQC'  # Emotion
destination = 'mamba_emotion_weights.pt'
# Construct the URL
url = f'https://drive.google.com/uc?id={file_id}'



# Download the file
gdown.download(url, destination, quiet=False)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


Downloading...
From (original): https://drive.google.com/uc?id=17J33ZY5lDiPneHy0ay29J5uxL-F0KAQC
From (redirected): https://drive.google.com/uc?id=17J33ZY5lDiPneHy0ay29J5uxL-F0KAQC&confirm=t&uuid=910e2f1d-9cdf-4e59-97de-c00429822559
To: /home/jovyan/mamba_emotion_weights.pt
100%|██████████| 517M/517M [00:06<00:00, 75.9MB/s] 


'mamba_emotion_weights.pt'

In [8]:
model.load_state_dict(
    torch.load('mamba_emotion_weights.pt', map_location=torch.device('cpu')),
    strict=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Make model explainable.
modified_model = ModifiedMambaForCausalLM(model, is_fast_forward_available=False)
modified_model.eval()
model.backbone.embeddings.requires_grad = False
pretrained_embeddings = model.backbone.embeddings

## Load dataset

Load SST-2 dataset.

In [9]:
validation_dataset = get_emotion_dataset(
    tokenizer=tokenizer,
    truncation=False,
    max_length=None
    )

/home/jovyan/venvs/hf_mamba_env/lib/python3.10/site-packages/datasets/load.py:1461: FutureWarning: The repository for philschmid/emotion contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/philschmid/emotion
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


## Generate explanation

Generate explanation for one sample.

In [85]:
# i = 533
i = 543
input_ids = validation_dataset.__getitem__(i)['input_ids'].unsqueeze(0).to(device)
label = torch.tensor(validation_dataset.__getitem__(i)['label']).long().to(device)
idx = torch.where(input_ids == 0)[1] + 1
pad_id = tokenizer.pad_token_id
pad_positions = (input_ids[0] == pad_id).nonzero(as_tuple=True)[0]

if len(pad_positions) > 0:
    idx = pad_positions[0]   # first padding token
    input_ids = input_ids[:, :idx]
    
embeddings = pretrained_embeddings(input_ids)

pad_id = torch.tensor([[tokenizer.pad_token_id]]).to(device)
baseline = pretrained_embeddings(pad_id)


# Standard LRP
R_lrp, prediction = relevance_propagation(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=6
)

# IG-LRP
R_ig_lrp, prediction = relevance_propagation_ig_lrp(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=6,
    baseline=baseline
)

## Visualization

For simplicity, we use the visualization utilities in Captum to display the results.

In [86]:
from captum.attr import visualization as viz

In [87]:
tokens = [
    tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens([id.item()])
    )
    for id in input_ids[0][1:-2]
]

attr_lrp = R_lrp[0][1:-2]
attr_lrp = attr_lrp / (np.abs(attr_lrp).max() + 1e-8)

attr_ig_lrp = R_ig_lrp[0][1:-2]
attr_ig_lrp = attr_ig_lrp / (np.abs(attr_ig_lrp).max() + 1e-8)

In [88]:
print("=== LRP ===")

viz.visualize_text([
    viz.VisualizationDataRecord(
        attr_lrp,
        torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
        torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
        true_class=label.item(),
        attr_class=label.item(),
        attr_score=attr_lrp.sum(),
        raw_input_ids=tokens,
        convergence_score=None
    )
])

print("=== IG-LRP ===")

viz.visualize_text([
    viz.VisualizationDataRecord(
        attr_ig_lrp,
        torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
        torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
        true_class=label.item(),
        attr_class=label.item(),
        attr_score=attr_ig_lrp.sum(),
        raw_input_ids=tokens,
        convergence_score=None
    )
])

print("done")

=== LRP ===


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
5,5 (3.89),5,1.17,| start of text |> i have spoken about before but the feeling is getting stronger and i am curious if others have


=== IG-LRP ===


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
5,5 (3.89),5,1.30,| start of text |> i have spoken about before but the feeling is getting stronger and i am curious if others have


done


In [71]:
def get_morf_curve_logits(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    with torch.no_grad():
        logits = model(temp_input_ids) 
        initial_logit = logits[0, -1, target_label].item()
        scores.append(initial_logit)

    for idx in sorted_indices:
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            logit = logits[0, -1, target_label].item()
            scores.append(logit)
            
    return scores
#morf_values = get_morf_curve_logits(modified_model, input_ids, sorted_indices, label.item())

In [72]:
def get_morf_curve_batched(model, embeddings, sorted_indices, target_label, chunk_size=8):
    seq_len = len(sorted_indices)
    base_embeddings = embeddings.clone().repeat(seq_len + 1, 1, 1)
    
    for i, idx in enumerate(sorted_indices):
        base_embeddings[i+1:, idx, :] = 0.0   # no +1, indices already correct
    
    all_logits = []
    with torch.no_grad():
        for i in range(0, base_embeddings.size(0), chunk_size):
            chunk = base_embeddings[i:i+chunk_size].to(device)
            logits = model(inputs_embeds=chunk)
            all_logits.extend(logits[:, -1, target_label].cpu().tolist())
    return all_logits

In [ ]:
import numpy as np
from tqdm import tqdm

num_samples = 1000
all_delta_afs = []

print(f"Evaluating {num_samples} samples from Emotion...")

pad_id = torch.tensor([[tokenizer.pad_token_id]]).to(device)
baseline = pretrained_embeddings(pad_id)

for i in tqdm(range(num_samples)):
    sample = validation_dataset.__getitem__(i)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    label = torch.tensor(sample['label']).long().to(device)
    
    pad_id = tokenizer.pad_token_id
    pad_positions = (input_ids[0] == pad_id).nonzero(as_tuple=True)[0]

    if len(pad_positions) > 0:
        idx = pad_positions[0]   # first padding token
        input_ids = input_ids[:, :idx]
        
    embeddings = pretrained_embeddings(input_ids)
    # Get pred_label PER SAMPLE
    with torch.no_grad():
        outputs = modified_model(inputs_embeds=embeddings)
        pred_label = torch.argmax(outputs[0, -1, :]).item()
    
    
    R, _ = relevance_propagation(
        model=modified_model, 
        embeddings=embeddings,
        targets=torch.tensor(pred_label).long().to(device), 
        n_classes=6
    )
    
    # R_ig_lrp, prediction = relevance_propagation_ig_lrp(
    #     model=modified_model,
    #     embeddings=embeddings,
    #     targets=label,
    #     n_classes=6,
    #     baseline=baseline
    # )
    
    attr = R[0]
    if isinstance(attr, np.ndarray):
        attr = torch.from_numpy(attr)
    
    idx_morf = torch.argsort(attr[1:].abs(), descending=True) + 1
    morf_curve = get_morf_curve_batched(modified_model, embeddings, idx_morf, pred_label)
    # af_morf = np.trapz(morf_curve) / (len(morf_curve) - 1)
    
    idx_lerf = torch.argsort(attr[1:].abs(), descending=False) + 1
    lerf_curve = get_morf_curve_batched(modified_model, embeddings, idx_lerf, pred_label)
    # af_lerf = np.trapz(lerf_curve) / (len(lerf_curve) - 1)
    
    x = np.linspace(0, 1, len(morf_curve))
    af_morf = np.trapz(morf_curve, x)   # AUC over [0,1]
    af_lerf = np.trapz(lerf_curve, x)
    # delta_af = af_morf - af_lerf
    delta_af = af_lerf - af_morf
    all_delta_afs.append(delta_af)
    # del R, embeddings, input_ids, attr, morf_curve, lerf_curve
    # gc.collect()
    # if i % 10 == 0:
    #     torch.cuda.empty_cache()
    

# Final Result
mean_delta_af = np.mean(all_delta_afs)
print(f"\nMean Delta AF over {num_samples} samples: {mean_delta_af:.4f}")
print(f"Paper Reference (Mamba-130M Emotion): 3.523")

Evaluating 1000 samples from Medical-Bios...


100%|██████████| 1000/1000 [15:00<00:00,  1.11it/s]


Mean Delta AF over 1000 samples: 3.5912
Paper Reference (Mamba-130M Medical-Bios): 3.906
